In [2]:
import cv2
import os
from ultralytics import YOLO

# --- CONFIGURATION ---
MODEL_PATH = 'runs/detect/train/weights/best.pt'
SOURCE_DIR = 'videos'
OUTPUT_DIR = 'detections/cropped_results'
model = YOLO(MODEL_PATH)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

video_files = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.mp4', '.avi', '.mov'))]

for video_name in video_files:
    input_path = os.path.join(SOURCE_DIR, video_name)
    output_path = os.path.join(OUTPUT_DIR, f"cropped_{video_name}")
    
    # Open the video to get properties
    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    
    # Calculate the top boundary (Bottom 2/5ths)
    # If height is 1000, 2/5 is 400. We want from 600 to 1000.
    y_start = int(height * (3/5)) 
    cropped_height = height - y_start

    # Set up Video Writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, cropped_height))

    print(f"Processing {video_name} (Bottom {cropped_height} pixels)...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # CROP: [y_start:y_end, x_start:x_end]
        cropped_frame = frame[y_start:height, 0:width]

        # Run inference on the CROP
        results = model.predict(cropped_frame, conf=0.50, verbose=False)
        
        # Plot the boxes onto the cropped frame
        annotated_frame = results[0].plot()

        # Write the frame to our new video
        out.write(annotated_frame)

    cap.release()
    out.release()
    print(f"Finished: {output_path}")

print("All videos processed!")

Processing Qualification 72 - 2025 Iowa Regional.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_Qualification 72 - 2025 Iowa Regional.mp4
Processing Qualification 79 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_Qualification 79 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4
Processing sf11m1.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_sf11m1.mp4
Processing Qualification 26 - 2025 Iowa Regional.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_Qualification 26 - 2025 Iowa Regional.mp4
Processing Qualification 15 - 2025 Iowa Regional.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_Qualification 15 - 2025 Iowa Regional.mp4
Processing f1m1.mp4 (Bottom 432 pixels)...
Finished: detections/cropped_results/cropped_f1m1.mp4
Processing FRC 2025 CTTD Quals 22 9990 5126 4455 v  1825 8825 3928 red